# Agentic Legacy Integrator for Mainframe-to-ML Migration
## 1. Legacy Input Analysis & Preprocessing

**Observation:**
The input is a legacy mainframe JCL (Job Control Language) script used for DFSMS (Data Facility Storage Management Subsystem) configuration. It contains several steps (`STEP021` to `STEP065`) that define storage groups, data classes, and storage classes for a DB2 database.

**Cleaning Strategy:**
1.  **Noise Reduction:** Filter out empty lines, JCL comments starting with `//*`, and generic system DD statements (like `SYSPRINT`, `SYSUDUMP`).
2.  **Step Identification:** Use regex to identify executable steps (e.g., `//STEP... EXEC`).
3.  **Command Parsing:** Extract the core `ISPSTART CMD(...)` blocks where the actual business logic resides.
4.  **Rule Extraction:** Use regex to parse parameters (e.g., `HIGHTHRS(85)`) into a structured JSON format (Key-Value pairs).

**Ambiguities:**
* Many parameters have empty parentheses (e.g., `MIGSYSNM()`). We need to decide whether to map these to `null` or omit them in the modern schema.
* The connection between `SCDS('CPAC.DFSMS.SCDS')` and the defined classes implies a relational dependency that needs to be preserved.

In [1]:
import json
import re

def parse_jcl_to_json(file_path):
    """
    Reads a JCL file line-by-line and extracts business rules cleanly.
    """
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()
    except FileNotFoundError:
        return f"Error: Could not find the file at {file_path}"

    structured_data = {
        "metadata": {
            "source_type": "Mainframe JCL",
            "description": "DFSMS Storage Management Rules"
        },
        "steps": []
    }

    current_step = None
    in_cmd = False
    cmd_buffer = ""

    for line in lines:
        # Step 1: Clean the line
        clean_line = line.strip()
        
        clean_line = re.sub(r'\[.*?\]', '', clean_line).strip()    

        if not clean_line or clean_line.startswith('//*'):
            continue

        # Step 2: Detect the Step Name (e.g., //STEP031 EXEC ...)
        step_match = re.match(r'^//(STEP\d+)\s+EXEC', clean_line)
        if step_match:
            current_step = step_match.group(1)
            continue

        # Step 3: Detect the start of a Command block
        if "CMD(" in clean_line:
            in_cmd = True
            cmd_buffer = clean_line.split("CMD(")[1]
            continue

        # Step 4: Accumulate parameters until the command closes
        if in_cmd:
            cmd_buffer += " " + clean_line
            if clean_line == ")" or (clean_line.endswith(")") and "+" not in clean_line):
                
                cmd_buffer = cmd_buffer.replace('+', ' ')
                
                action_match = re.search(r'(DEFINE|DISPLAY|ALTER)', cmd_buffer)
                action = action_match.group(1) if action_match else "UNKNOWN"
                
                params = re.findall(r'([A-Z0-9]+)\((.*?)\)', cmd_buffer)
                rules = {k: (v.strip() if v.strip() else None) for k, v in params}
                
                structured_data["steps"].append({
                    "step_name": current_step or "UNKNOWN_STEP",
                    "action": action,
                    "extracted_rules": rules
                })
                
                in_cmd = False
                cmd_buffer = ""

    return structured_data

# --- Execution ---
file_path = '../data/raw/sms'
parsed_rules = parse_jcl_to_json(file_path)

if isinstance(parsed_rules, str):
    print(parsed_rules)
else:
    with open('../data/processed/structured_rules.json', 'w') as out_file:
        json.dump(parsed_rules, out_file, indent=4)
        
    print("Parsing Successful! Here is a preview of the extracted rules:\n")
    print(json.dumps(parsed_rules, indent=4)[:800] + "\n...\n}")

Parsing Successful! Here is a preview of the extracted rules:

{
    "metadata": {
        "source_type": "Mainframe JCL",
        "description": "DFSMS Storage Management Rules"
    },
    "steps": [
        {
            "step_name": "STEP031",
            "action": "DEFINE",
            "extracted_rules": {
                "SCDS": "'CPAC.DFSMS.SCDS'",
                "STORGRP": "SGIXXX",
                "DESCR": "STORAGE GROUP FOR DB2",
                "AUTOMIG": "N",
                "MIGSYSNM": null,
                "AUTOBKUP": "N",
                "BKUPSYS": null,
                "AUTODUMP": null,
                "DMPSYSNM": null,
                "OVRFLOW": null,
                "EXTSGNM": null,
                "DUMPCLAS": null,
                "HIGHTHRS": "85",
                "LOWTHRS": "30",
                "GUARBKFR": "NOLIMIT",
             
...
}
